In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))  # nếu notebook ở training/
from training.train import train_model
from training.loss import get_loss
from training.scheduler import get_scheduler


In [2]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets.preprocessing import load_captions, filter_valid_images, flatten_data
from datasets.vocab import Vocabulary
from datasets.dataset import FlickrDataset
from datasets.collate import collate_fn
from models.cnn_encoder import CNNEncoder
from models.transformer_encoder import TransformerEncoder
from models.transformer_decoder import TransformerDecoder
from models.caption_model import CaptionModel
from training.loss import get_loss
from training.scheduler import get_scheduler

root_dir = os.path.abspath("..")
if not os.path.exists(os.path.join(root_dir, "data")):
    root_dir = os.path.abspath(".")

data_dir = os.path.join(root_dir, "data")
image_dir = os.path.join(data_dir, "Images")
captions_file = os.path.join(data_dir, "captions.txt")

captions_dict = load_captions(captions_file)
captions_dict = filter_valid_images(image_dir, captions_dict)
image_paths, captions = flatten_data(image_dir, captions_dict)

vocab = Vocabulary(freq_threshold=5)
vocab.build_vocab(captions)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

dataset = FlickrDataset(image_paths, captions, vocab, transform=transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
 )

batch_size = 32
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn
 )
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn
 )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

max_len = max(len(vocab.numericalize(c)) + 2 for c in captions)
encoder = CNNEncoder(embed_dim=512)
trans_enc = TransformerEncoder(embed_dim=512, num_heads=2)
decoder = TransformerDecoder(
    vocab_size=len(vocab),
    embed_dim=512,
    num_head=2,
    ff_dim=512,
    max_len=max_len
 )
model = CaptionModel(encoder, trans_enc, decoder).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = get_loss()
scheduler = get_scheduler(optimizer)

c:\Users\ADMIN\anaconda3\envs\BaiTapFlickr8k\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ADMIN\anaconda3\envs\BaiTapFlickr8k\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
import glob
import os

config = {
    "epochs": 10,
    "batch_size": 32,
    "lr": 1e-4,
    "embed_dim": 512,
    "wandb_mode": "online"  # local logging, khong can login
}

ckpt_dir = "checkpoints"
ckpt_list = sorted(glob.glob(os.path.join(ckpt_dir, "epoch_*.pt")))
resume_path = ckpt_list[-1] if ckpt_list else None

train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    device=device,
    epochs=config["epochs"],
    config=config,
    resume_path=resume_path,
    checkpoint_dir=ckpt_dir,
    use_wandb=True,
    wandb_mode=config["wandb_mode"],
    wandb_project="image-captioning",
    log_images=True
 )

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\ADMIN\_netrc.
wandb: Currently logged in as: phucga150625 (phucga15062005) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


KeyboardInterrupt: 